# 🔒 Privacy Filter — Fine-tuning su Kaggle (T4)
**openai/privacy-filter** — Modello generico italiano per anonimizzazione PII

## ⚙️ Prerequisiti Kaggle

**⚠️ IMPORTANTE — VERIFICA TELEFONO**: Internet su Kaggle funziona **solo se il tuo account è verificato via telefono**. Senza verifica, il `pip install git+...` fallisce con `exit code 128`.

Verifica: `kaggle.com → avatar (alto dx) → Settings → Phone verification → numero + codice SMS`.

Prima di eseguire:
1. Account verificato via telefono (vedi sopra)
2. **Settings (pannello destro del notebook) → Accelerator** → `GPU T4 x2` (o `GPU T4`)
3. **Settings → Internet** → ON
4. **Session** → almeno 30 GB di RAM (il default basta)

## ℹ️ Note sulla versione
Questo notebook usa un **unico modello generico italiano** (non più 2-step docs+legale).
Lo step legale è stato rimosso perché overfittava: classificava `Mario Rossi` come `parte_in_causa` anche fuori contesto legale.

## Output
Il checkpoint finale va in `/kaggle/working/checkpoint_generic_italian/` — scaricabile come zip dal pannello **Output**.

---
## Indice
1. Setup: install + scrittura `dataset_builder.py`
2. Verifica GPU CUDA (T4)
3. Genera dataset generico italiano (3200 esempi: train+val+test)
4. Training modello generico (15 epoche, ~30 min)
5. Test qualitativo
6. Valutazione quantitativa (precision/recall/F1) su test set held-out
7. Zip checkpoint per download
8. Inferenza custom sul tuo testo


## 1. Setup

Installa `opf` e scrive il modulo `dataset_builder.py` in `/kaggle/working/` (inline, così il notebook è self-contained e non servono upload esterni).

In [ ]:
# Cella 1 — Diagnostica + Install opf + scrittura dataset_builder.py
import subprocess, sys, os, base64, urllib.request

os.chdir('/kaggle/working')

print('🔍 Diagnostica Internet...')
try:
    urllib.request.urlopen('https://github.com', timeout=5)
    print('✅ Internet: OK')
except Exception as e:
    print(f'❌ Internet NON disponibile: {e}')
    print('📋 Verifica account telefono e toggle Internet')
    raise SystemExit(1)

print('📦 Installazione opf...')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'git+https://github.com/openai/privacy-filter.git',
    'accelerate', 'safetensors', 'tiktoken'])
print('✅ opf installato')

DATASET_BUILDER_B64 = (
    "..."
    "cmV0dXJuIHBhdGhzCg=="
)
src_code = base64.b64decode(DATASET_BUILDER_B64).decode('utf-8')

with open('/kaggle/working/dataset_builder.py', 'w') as f:
    f.write(src_code)

if '/kaggle/working' not in sys.path:
    sys.path.insert(0, '/kaggle/working')

from dataset_builder import LABEL_SPACE, build_complete_dataset
n_labels = len(LABEL_SPACE['span_class_names'])
print(f'✅ dataset_builder.py scritto ({n_labels} label, {len(src_code):,} caratteri)')

## 2. Verifica GPU CUDA

Su Kaggle con GPU T4: `DEVICE_CLI='cuda'`, la T4 ha 16 GB VRAM dedicati.

In [ ]:
# Cella 2 — Verifica device e ambiente
import torch, sys

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in __import__('os').environ

if torch.cuda.is_available():
    DEVICE_CLI = 'cuda'
    DEVICE_HF  = 0
    print(f'✅ CUDA: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'   CUDA count: {torch.cuda.device_count()}')
else:
    DEVICE_CLI = 'cpu'
    DEVICE_HF  = -1
    print('⚠️  GPU non rilevata — verifica Settings → Accelerator → GPU T4 x2')

print(f'\nAmbiente: {"Kaggle" if IS_KAGGLE else "altro"}')
print(f'PyTorch: {torch.__version__}')


## 3. Genera dataset sintetico (solo Step 1 — generico italiano)

**Cambio di strategia dalla versione v2**: Step 2 (dominio legale specializzato) è stato **rimosso**. Nei test aggiungeva rumore: riclassificava `private_person → parte_in_causa` fuori contesto legale, compromettendo l'uso generico.

Ora il training si concentra su un **unico step "generico italiano"** con 3200 esempi che coprono:
- Documenti (CF, CI, patente, passaporto, P.IVA, IBAN, tessera sanitaria)
- Email, messaggi, chat informale
- CV, presentazioni, biografie brevi
- News, articoli, cronache
- Comunicazioni business, fatture, ordini, candidature
- Bolletta, estratti conto, certificati


In [ ]:
# Cella 3 — Genera dataset generico italiano (3200 esempi, 3 file .jsonl)
from dataset_builder import build_complete_dataset, write_splits_to_disk, validate_spans, label_distribution

bundle = build_complete_dataset(
    n_step1=(2400, 400, 400),    # train, val, test
    seed=42,                      # riproducibilità
    negative_rate=0.20,           # 20% esempi negativi (riduce falsi positivi)
    include_step2=False,          # step2 legale disabilitato (aggiungeva rumore)
)

TRAIN = bundle['step1']['train']
VAL   = bundle['step1']['val']
TEST  = bundle['step1']['test']

# Validazione
tot_err = 0
for name, data in [('train', TRAIN), ('val', VAL), ('test', TEST)]:
    err = validate_spans(data, name)
    tot_err += err
    neg = sum(1 for ex in data if not ex['spans'])
    print(f'  {name:5s}: {len(data):>4} esempi, {neg} neg ({100*neg/len(data):.0f}%), {err} errori')

if tot_err != 0:
    raise RuntimeError(f'{tot_err} errori nel dataset')

# Salva file .jsonl (solo step1)
paths = write_splits_to_disk(bundle, base_dir='/kaggle/working/datasets')
print('\nFile salvati:')
for key, (path, n, size) in paths.items():
    print(f'  {path}  ({n} esempi, {size/1024:.1f} KB)')

print('\nDistribuzione label (train):')
for label, cnt in sorted(label_distribution(TRAIN).items(), key=lambda x: -x[1]):
    print(f'  {label:25s}: {cnt}')


## 4. Training — Modello generico italiano

Fine-tuning del modello base `openai/privacy-filter` sui 2400 esempi generici italiani. Unico step (non più due).

**Parametri Kaggle T4**: batch=1 + grad_accum=4, 15 epoche. Tempo stimato: 20-40 min.


In [ ]:
# Cella 4 — Training modello generico italiano
import subprocess, os, json
from dataset_builder import LABEL_SPACE

LABEL_SPACE_PATH = '/kaggle/working/custom_label_space.json'
with open(LABEL_SPACE_PATH, 'w') as f:
    json.dump(LABEL_SPACE, f, indent=2)
print(f'✅ Label space scritto ({len(LABEL_SPACE["span_class_names"])} categorie)')

OUTPUT_DIR = '/kaggle/working/checkpoint_generic_italian'
NUM_EPOCHS = 15
BATCH_SIZE = 1

cmd = [
    'opf', 'train', '/kaggle/working/datasets/step1_train.jsonl',
    '--validation-dataset', '/kaggle/working/datasets/step1_val.jsonl',
    '--label-space-json', LABEL_SPACE_PATH,
    '--output-dir', OUTPUT_DIR,
    '--epochs', str(NUM_EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--grad-accum-steps', '4',
    '--device', DEVICE_CLI,
]

print('🚀 Training modello generico italiano...')
print('Comando:', ' '.join(cmd))
print('=' * 60)

process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                           text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
process.wait()

if process.returncode == 0:
    print('\n✅ Training completato:', OUTPUT_DIR)
    FINAL_CHECKPOINT = OUTPUT_DIR
else:
    print(f'\n❌ Training fallito (exit code {process.returncode})')


## 5. Test qualitativo

Esegue il modello finetuned su frasi di esempio per mostrare visivamente come riconosce le entità italiane.

In [ ]:
# Test qualitativo del modello finetuned (usa API opf.OPF, non transformers.pipeline)
from opf import OPF
import os

print(f'Caricamento modello da: {FINAL_CHECKPOINT}')
opf_model = OPF(
    model=FINAL_CHECKPOINT,
    device=DEVICE_CLI,         # 'cuda' o 'cpu' (MPS non usato in inference qui)
    output_mode='typed',       # mantiene le label originali (non collassa a [REDACTED])
    decode_mode='viterbi',     # migliore qualità di argmax
)
print('✅ Modello caricato\n')

# Wrapper che emula l'API di transformers.pipeline (entity_group/start/end/score)
# Così le funzioni di valutazione quantitative restano le stesse.
def classify(text):
    result = opf_model.redact(text)
    return [
        {'entity_group': s.label, 'start': s.start, 'end': s.end,
         'word': s.text, 'score': getattr(s, 'score', 1.0)}
        for s in result.detected_spans
    ]

TEST_CASES = [
    "Il sottoscritto Mario Rossi, codice fiscale RSSMRA80A01H501U, chiede il rilascio del certificato.",
    "Carta d'identità n. AB1234567, intestata a Giulia Bianchi, nata a Milano il 15/03/1990.",
    "Patente: CD456EF78901G — titolare: Luca Ferrari, residente a Roma.",
    "Bonifico su IBAN IT60X0542811101000000123456 intestato a Marco Ricci.",
    "Avanti a me, Notaio dott. Carlo Verdi, sono comparsi: il sig. Antonio Russo, C.F. RSSNTN75C15F205K.",
    "Nel procedimento n. 1234/2023 RG pendente avanti al Tribunale di Roma, tra Francesca Mancini (attore) e Roberto Costa (convenuto).",
    "L'immobile censito come foglio 45, mappale 123, sub. 7, sito nel Comune di Napoli.",
]

print('=' * 70)
for i, text in enumerate(TEST_CASES, 1):
    result = opf_model.redact(text)
    print(f'[{i:2d}] Input   : {text}')
    print(f'     Redatto : {result.redacted_text}')
    for s in result.detected_spans:
        print(f'       • {s.label:25s} {s.text!r}  ({s.start}:{s.end})')
    print()


## 6. Valutazione quantitativa sul test set

Precision / Recall / F1 per ogni categoria su 450 esempi di test mai visti durante il training.

Soglia indicativa: **F1 > 0.85** per categorie principali.

In [ ]:
# Metriche precision/recall/F1 su test set held-out (usa la fn classify() definita sopra)
from collections import defaultdict

def compute_span_metrics(classifier_fn, test_data, match='exact'):
    tp = defaultdict(int); fp = defaultdict(int); fn = defaultdict(int)
    for ex in test_data:
        text = ex['text']
        gold = set()
        for label, spans in ex.get('spans', {}).items():
            for s, e in spans:
                gold.add((label, s, e))
        preds = set()
        for r in classifier_fn(text):
            preds.add((r['entity_group'], int(r['start']), int(r['end'])))
        if match == 'exact':
            for p in preds:
                (tp if p in gold else fp)[p[0]] += 1
            for g in gold:
                if g not in preds:
                    fn[g[0]] += 1
        else:
            used_gold = set()
            for p_lab, p_s, p_e in preds:
                hit = False
                for g in gold - used_gold:
                    g_lab, g_s, g_e = g
                    if g_lab == p_lab and not (p_e <= g_s or g_e <= p_s):
                        tp[p_lab] += 1; used_gold.add(g); hit = True; break
                if not hit:
                    fp[p_lab] += 1
            for g_lab, g_s, g_e in gold - used_gold:
                fn[g_lab] += 1
    return tp, fp, fn

def print_metrics_table(tp, fp, fn, title=''):
    labels = sorted(set(tp) | set(fp) | set(fn))
    tot_tp = sum(tp.values()); tot_fp = sum(fp.values()); tot_fn = sum(fn.values())
    print(f'\n{title}')
    print(f'{"label":25s} {"TP":>5s} {"FP":>5s} {"FN":>5s} {"P":>7s} {"R":>7s} {"F1":>7s}')
    print('─' * 70)
    for l in labels:
        p = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] > 0 else 0.0
        r = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] > 0 else 0.0
        f1 = 2*p*r/(p+r) if p+r > 0 else 0.0
        marker = ' ✅' if f1 >= 0.85 else (' ⚠️ ' if f1 >= 0.7 else ' ❌')
        print(f'{l:25s} {tp[l]:>5d} {fp[l]:>5d} {fn[l]:>5d} {p:>7.3f} {r:>7.3f} {f1:>7.3f}{marker}')
    mp = tot_tp / (tot_tp + tot_fp) if tot_tp + tot_fp > 0 else 0.0
    mr = tot_tp / (tot_tp + tot_fn) if tot_tp + tot_fn > 0 else 0.0
    mf1 = 2*mp*mr/(mp+mr) if mp+mr > 0 else 0.0
    print('─' * 70)
    print(f'{"MICRO AVG":25s} {tot_tp:>5d} {tot_fp:>5d} {tot_fn:>5d} {mp:>7.3f} {mr:>7.3f} {mf1:>7.3f}')
    return mf1

print('Match EXACT (span perfettamente coincidenti):\n')
tp, fp, fn = compute_span_metrics(classify, TEST, match='exact')
f1_exact = print_metrics_table(tp, fp, fn, title=f'═══ TEST — Modello generico italiano ({len(TEST)} esempi) ═══')

print('\n\nMatch OVERLAP (più permissivo):')
tp_o, fp_o, fn_o = compute_span_metrics(classify, TEST, match='overlap')
f1_overlap = print_metrics_table(tp_o, fp_o, fn_o, title='═══ TEST — Overlap ═══')

print(f'\n{"═"*70}')
print(f'F1 exact: {f1_exact:.3f}  |  F1 overlap: {f1_overlap:.3f}')
print(f'{"═"*70}')


## 7. Scarica il checkpoint

I checkpoint sono già in `/kaggle/working/` — visibili nel pannello **Output** a destra.

Per scaricarli:
1. Sulla destra, clicca sul pannello "Output"
2. Naviga in `checkpoint_step2_legal/` (il finale) o `checkpoint_step1_italian_docs/` (intermedio)
3. Download dei file `model.safetensors`, `config.json`, `finetune_summary.json`, `USAGE.txt`

Per usarli localmente sul Mac: copia l'intera cartella in `~/Documents/privacy_filter_checkpoints/step2_legal/` e caricala con `transformers.pipeline('token-classification', model='~/Documents/privacy_filter_checkpoints/step2_legal')`.

Se preferisci uno zip unico:

In [ ]:
# Cella 6 — Zip del checkpoint finale per download
import shutil, os

if os.path.isdir(FINAL_CHECKPOINT):
    zip_path = '/kaggle/working/checkpoint_generic_italian.zip'
    shutil.make_archive('/kaggle/working/checkpoint_generic_italian', 'zip', FINAL_CHECKPOINT)
    size = os.path.getsize(zip_path) / 1e6
    print(f'✅ {zip_path}  ({size:.1f} MB)')
    print('\nScarica lo zip dal pannello Output di Kaggle (pannello destro).')
else:
    print(f'⚠️  {FINAL_CHECKPOINT} non trovato')


## 8. Inferenza custom — analizza i tuoi testi

Usa il modello finetuned per analizzare qualsiasi testo. Modifica la lista `MY_TEXTS` con le frasi o documenti che vuoi processare.

La cella è **self-contained**: se hai riavviato il kernel, ricarica automaticamente il modello dall'ultimo checkpoint disponibile (`step2_legal` se esiste, altrimenti `step1_italian_docs`).

Per analizzare un intero file: usa la variante `analyze_file('/kaggle/input/...')`.

In [ ]:
# Cella 9 — Inferenza custom sul tuo testo

# ═════════════════════════════════════════════════════════════════
# INSERISCI QUI I TUOI TESTI — modifica questa lista
# ═════════════════════════════════════════════════════════════════
MY_TEXTS = [
    "Il sottoscritto Mario Rossi, CF RSSMRA80A01H501U, residente in Via Roma 10, Milano, presenta ricorso avanti al Tribunale di Roma.",
    "Per bonifici usare IBAN IT60X0542811101000000123456 intestato a Luigi Bianchi.",
    "Avanti a me, Notaio dott. Carlo Verdi, è comparso il sig. Antonio Russo, nato il 15/03/1980 a Napoli.",
    # Aggiungi qui i tuoi testi (frasi, paragrafi, documenti)...
]

# ═════════════════════════════════════════════════════════════════
# Safety: ricarica il modello se non presente (kernel riavviato)
# ═════════════════════════════════════════════════════════════════
import os

try:
    opf_model
    print('✅ opf_model già caricato (riuso)')
except NameError:
    from opf import OPF
    CHECKPOINT = ('/kaggle/working/checkpoint_step2_legal'
                  if os.path.isdir('/kaggle/working/checkpoint_step2_legal')
                  else '/kaggle/working/checkpoint_step1_italian_docs')
    if not os.path.isdir(CHECKPOINT):
        raise RuntimeError('Nessun checkpoint trovato. Esegui prima le celle di training.')
    print(f'⚙️  Ricarica modello da {CHECKPOINT}...')
    opf_model = OPF(
        model=CHECKPOINT,
        device='cuda' if __import__('torch').cuda.is_available() else 'cpu',
        output_mode='typed',
        decode_mode='viterbi',
    )
    print('✅ Modello caricato')

# ═════════════════════════════════════════════════════════════════
# Funzioni di analisi
# ═════════════════════════════════════════════════════════════════
def analyze(text, model=None, verbose=True):
    """Analizza un testo e (se verbose) stampa entità rilevate + testo redatto.
    Ritorna il RedactionResult di opf.
    """
    if model is None:
        model = opf_model
    result = model.redact(text)

    if verbose:
        print(f'📝 Input:\n   {text}\n')
        if result.detected_spans:
            print(f'🔍 {len(result.detected_spans)} entità rilevate:')
            for s in result.detected_spans:
                score = ''
                if getattr(s, 'score', None) is not None:
                    score = f'  (confidence={s.score:.2f})'
                print(f'   • {s.label:25s} {s.text!r:40s} [{s.start}:{s.end}]{score}')
        else:
            print('🔍 Nessuna entità rilevata (testo già pulito)')
        print(f'\n✅ Testo redatto:\n   {result.redacted_text}')
    return result


def analyze_file(file_path, chunk_chars=4000):
    """Legge un file di testo e lo analizza a chunk (utile per documenti lunghi).

    chunk_chars: dimensione massima di ogni chunk in caratteri. Il modello ha un
    contesto massimo — chunk troppo grandi possono essere tronati.
    """
    with open(file_path, encoding='utf-8') as f:
        full_text = f.read()
    print(f'📄 File: {file_path}  ({len(full_text):,} caratteri)')
    print(f'   Chunking a {chunk_chars} caratteri per chunk\n')

    all_spans = []
    offset = 0
    redacted_parts = []
    for i in range(0, len(full_text), chunk_chars):
        chunk = full_text[i:i + chunk_chars]
        result = analyze(chunk, verbose=False)
        # Offsetta gli span alla posizione globale
        for s in result.detected_spans:
            s_copy = type('Span', (), {
                'label': s.label, 'text': s.text,
                'start': s.start + offset, 'end': s.end + offset,
                'score': getattr(s, 'score', None),
            })()
            all_spans.append(s_copy)
        redacted_parts.append(result.redacted_text)
        offset += len(chunk)

    print(f'🔍 Totale entità rilevate: {len(all_spans)}')
    from collections import Counter
    dist = Counter(s.label for s in all_spans)
    for label, cnt in sorted(dist.items(), key=lambda x: -x[1]):
        print(f'   {label:25s}: {cnt}')
    print(f'\n✅ Testo redatto salvato in /kaggle/working/output_redatto.txt')
    with open('/kaggle/working/output_redatto.txt', 'w', encoding='utf-8') as f:
        f.write(''.join(redacted_parts))
    return all_spans


# ═════════════════════════════════════════════════════════════════
# Esecuzione: analizza MY_TEXTS
# ═════════════════════════════════════════════════════════════════
for i, text in enumerate(MY_TEXTS, 1):
    print(f'\n{"═"*70}\nESEMPIO {i}\n{"═"*70}')
    analyze(text)

# ─── ESEMPIO USO CON FILE (decommenta per usarlo) ───
# Se carichi un file .txt via "Add Data" su Kaggle, sarà in /kaggle/input/<nome-dataset>/file.txt
# all_spans = analyze_file('/kaggle/input/mio-dataset/documento.txt')
